In [1]:
from sentence_transformers import SentenceTransformer

# Load the Sentence-Transformer model
MODEL = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL)

In [2]:
import pandas as pd

# Load the dataset
dataset = pd.read_csv("data/Mastodon_instances_metadata_v3.csv")

# Drop rows with missing values in both columns: keep only rows with some kind of descriptions
df = dataset.dropna(subset=["description", "extended_description"])

In [3]:
df.describe()

,active_users
count,2125.000000
mean,332.414118
std,5980.511035
min,1.000000
25%,3.000000
50%,13.000000
75%,67.000000
max,269627.000000


In [4]:
import os
import pickle

embedding_cache_path = "embeddings/title_emb.pkl"

instances = df['instance'].tolist()
print(instances)

embedding_dict = {}

if not os.path.exists(embedding_cache_path):
    embeddings = model.encode(instances, show_progress_bar=True, convert_to_numpy=True)

    embedding_dict = {
        instances[i]: embeddings[i] 
        for i in range(len(instances))
    }

    with open(embedding_cache_path, "wb") as fOut:
        pickle.dump(embedding_dict, fOut)

else:
    with open(embedding_cache_path, "rb") as fIn:
        embedding_dict = pickle.load(fIn)

['leipzig.town', 'v2br.social', 'social.bund.de', 'mstdn.lt', 'indg.club', 'greennuclear.online', 'hoosier.social', 'philately.social', 'gamedev.berlin', 'kzoo.to', 'solarpunk.moe', 'greennet.social', 'social.webitel.me', 'social.sciences.re', 'mastodon.macstories.net', 'birdbutt.com', 'dagegenbewegen.social', 'strangeplace.me', 'allpeep.social', 'pon.ee', 'ecologie.social', 'grants.cafe', 'mastodon.africa', 'jbo.social', 'games.ngo', 'recurse.social', 'm.aqr.af', 'livester.net', 'attractive.space', 'graz.social', 'konfigurationsmanufaktur.de', 'mastodon.tuxtendo.nl', 'mas.to', 'muenchen.social', 'masto.nyc', 'social.la10cy.net', 'm6n.onsen.tech', 'toot.community', 'friendica.utzer.de', 'fedinerds.social', 'mathtod.online', 's.pebcak.de', 'social.medlab.host', 'wptoots.social', 'lor.sh', 'sauropods.win', 'social.easya.solutions', 'mastodon.wien', 'metalverse.social', 'thelife.boats', 'nrkn.fr', 'e.fo', 'isosceles.monster', 'm.ai6yr.org', 'imd.social', 'legal.social', 'moe.cat', 'toot.w

Batches:   0%|          | 0/67 [00:00<?, ?it/s]

In [5]:
instance = "planetearth.social"
embedding = embedding_dict[instance]

print(f"Instance: {instance}, Embeddings: {embedding}")

Instance: planetearth.social, Embeddings: [ 8.83060470e-02  2.54243203e-02  3.41729494e-03  1.98231302e-02
  2.78568305e-02 -2.00446416e-02 -9.40849655e-04 -6.17956892e-02
 -9.35982075e-03  2.77521219e-02  3.03075779e-02 -3.40119824e-02
  3.31364311e-02 -2.58127991e-02 -3.27193402e-02 -3.96002717e-02
 -8.00678059e-02 -5.38920239e-02  4.87573929e-02 -6.79888157e-03
 -7.28336349e-02  8.12856108e-03 -2.15709466e-03  8.13277960e-02
  1.05070742e-02  5.32756113e-02 -7.07428753e-02  2.00342406e-02
  4.02307063e-02  2.93727256e-02  1.96814761e-02  7.83096775e-02
  3.04991193e-02  5.05836606e-02 -6.15814403e-02  5.96756116e-02
  1.84200052e-02 -1.07249366e-02 -6.26789629e-02 -1.30798703e-03
  1.65042989e-02 -6.16490804e-02  5.35350554e-02 -6.44467622e-02
 -3.94030809e-02  1.11833252e-02 -2.18077525e-02 -3.82811129e-02
  1.47402612e-02  2.46514776e-03 -2.32548099e-02 -4.72049341e-02
 -8.40054359e-03  4.91185971e-02 -2.20396761e-02 -1.25851342e-03
 -6.65890425e-03  2.36616493e-03  4.79216576e-02

In [6]:
import numpy as np


def find_most_similar(query_embedding, embeddings, instances, top_k=10):
    """Find the most similar embeddings to a query embedding and return their names"""
    # Calculate cosine similarity (dot product for normalized embeddings)
    similarities = np.dot(embeddings, query_embedding)
    top_indices = np.argsort(similarities)[-top_k-1:][::-1]
    
    # Return the instance names and their similarity scores
    top_instances = [instances[idx] for idx in top_indices][1::]
    top_similarities = similarities[top_indices][1::]
    
    return top_instances, top_similarities

res = find_most_similar(embedding, embeddings, instances)

print(res)

(['cosmos.social', 'astronomy.social', 'turtleisland.social', 'solarsystem.social', 'qth.social', 'astrodon.social', 'blueplanet.social', 'gardenstate.social', 'eworld.social', 'neos.social'], array([0.66528946, 0.65092117, 0.6460783 , 0.6373582 , 0.63513076,
       0.6326522 , 0.6250671 , 0.60761327, 0.6062823 , 0.6021239 ],
      dtype=float32))
